# 01 · Quickstart: the CRAF'd Forecast API

**What this teaches:** how to get CRAF'd's conflict forecasts and the recorded deaths from the API: the bulk download (the recommended way), slices (a smaller piece), the three levels (grid cell, admin 1, country), and how to read the summary columns.

The API never serves the model's individual draws, only summaries of them. Every column is explained in the [data dictionary](../docs/api/data_dictionary.md); the address, the key and the options are in the [partner guide](../docs/api/README.md).

**You need an API key** (set in `.env`, see `notebooks/README.md`).

## 1 · Setup

In [ ]:
import io
import os

import pandas as pd
import requests
from dotenv import load_dotenv

from views_crafdapi.client import CrafdApiClient

load_dotenv()  # reads APPWRITE_DATASTORE_API_KEY (your key) and, optionally, VIEWS_API_URL

BASE_URL = os.environ.get("VIEWS_API_URL", "https://crafdapi.viewsforecasting.org")
API_KEY = os.environ.get("APPWRITE_DATASTORE_API_KEY", "").strip().strip('"')
HEADERS = {"X-API-Key": API_KEY}  # for the bulk download, which is a plain file request

client = CrafdApiClient(BASE_URL, API_KEY, timeout=300)
print("Base URL:", BASE_URL)
print("API key :", "set" if API_KEY else "MISSING: set it in .env")


def download_bulk(level, outcome):
    """One bulk file as a DataFrame, and the month the forecast was made in. A large file over a
    slow connection can time out or drop; one retry covers that."""
    for attempt in (1, 2):
        try:
            resp = requests.get(f"{BASE_URL}/bulk/{level}/{outcome}", headers=HEADERS, timeout=600)
            break
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError,
                requests.exceptions.ChunkedEncodingError):
            if attempt == 2:
                raise
            print("The download was cut off; trying once more...")
    if not resp.ok:   # show the API's own explanation, not only the status code
        raise RuntimeError(f"API error {resp.status_code} on /bulk/{level}/{outcome}: {resp.text[:300]}")
    return pd.read_parquet(io.BytesIO(resp.content)), resp.headers["X-Forecast-Made-In"]

In [ ]:
# Stop early, with a clear message, when there is no usable key: otherwise the first data call
# fails with a 401 that does not say the .env is missing.
if not API_KEY or API_KEY == "your-api-key-here":
    raise SystemExit(
        "No usable API key. Copy .env.example to .env at the root of this repository, set\n"
        "APPWRITE_DATASTORE_API_KEY to your key (ask VIEWS for one), restart the kernel and\n"
        "run again. 03_offline_demo.ipynb needs no key."
    )

# /health also asks the key store; if only that fails, the data may still be served, so go on.
try:
    health = client.health()
except RuntimeError as e:
    health = {}
    print("Health check unavailable, continuing:", e)
else:
    print("Service:", health["status"], "| forecast loaded:", health.get("forecast_loaded"))
if health.get("forecast_loaded") is False:
    raise SystemExit("The service has no forecast loaded yet (it answers 503 to data requests). "
                     "Try again in a few minutes.")

## 2 · The bulk download: one file per outcome and level

`GET /bulk/{level}/{outcome}` returns one ready-made Parquet file: every place and every month. Levels are `grid`, `admin1` and `country`; outcomes are `sb` (state-based), `ns` (non-state) and `os` (one-sided violence). The month the forecast was made in comes in the `X-Forecast-Made-In` header.

In [ ]:
admin1, FORECAST_MADE_IN = download_bulk("admin1", "sb")
print("Forecast made in:", FORECAST_MADE_IN)
print("Rows:", len(admin1), "| columns:", len(admin1.columns))
admin1.columns.tolist()

## 3 · Recorded months and forecast months

Each row is one place in one month. A **recorded** month (before the forecast starts) has the recorded number of deaths in `sb_actual`, and empty summary columns. A **forecast** month has the summary columns, and an empty `sb_actual`. The month decides the kind: the forecast starts in the month it was made in (the `X-Forecast-Made-In` header), so every row before it is recorded and every row from it on is a forecast. (A forecast row whose draws were all unusable has empty summaries too; splitting by month still places it.)

In [ ]:
forecast_rows = admin1[admin1["month"] >= FORECAST_MADE_IN]
recorded_rows = admin1[admin1["month"] < FORECAST_MADE_IN]
print("Forecast months:", forecast_rows["month"].min(), "to", forecast_rows["month"].max())
print("Recorded months:", recorded_rows["month"].min(), "to", recorded_rows["month"].max())
FIRST_FORECAST_MONTH = forecast_rows["month"].min()

## 4 · Reading the summary columns

For a forecast month, each outcome has: the chances of more than 0, 25, 100, 500 and 1,000 deaths (`sb_p_gt0` … `sb_p_gt1000`, between 0 and 1), the expected number of deaths (`sb_mean`), the deaths if there are any (`sb_mean_if_any`, `sb_median_if_any`), a severe case (`sb_severe_scenario`: the average of the worst 5% of draws), and `n_draws`, how many draws the summaries come from.

Here: the ten admin-1 regions with the highest chance of more than 25 deaths in the first forecast month.

In [ ]:
first = forecast_rows[forecast_rows["month"] == FIRST_FORECAST_MONTH]
first.nlargest(10, "sb_p_gt25")[
    ["admin1_name", "country_name", "sb_p_gt0", "sb_p_gt25", "sb_mean", "sb_severe_scenario", "n_draws"]
]

## 5 · Slices: a smaller piece, as JSON

`GET /slice/{level}/{outcome}?places=...&months=...&columns=...`, or `client.slice(level, outcome, places=..., months=..., columns=...)`. Places are named by their id at that level: the PRIO-GRID cell id, the GAUL admin-1 code, or the GAUL country code (not ISO3). Months are written `2027-01`.

To find a country's GAUL code, look it up in any file or slice at country level, which also carries the ISO3 code:

In [ ]:
countries = client.slice("country", "sb", months=[FIRST_FORECAST_MONTH], columns=["sb_mean"])
NIGERIA = int(countries.loc[countries["country_iso3"] == "NGA", "country_code"].iloc[0])
print("Nigeria's GAUL country code:", NIGERIA)

In [ ]:
# Nigeria, the first six forecast months, a few columns.
months = sorted(forecast_rows["month"].unique())[:6]
client.slice("country", "sb", places=[NIGERIA], months=months,
             columns=["sb_p_gt25", "sb_mean", "sb_severe_scenario"])

In [ ]:
# The recorded deaths of the last twelve recorded months, for the same country.
last_recorded = sorted(recorded_rows["month"].unique())[-12:]
client.slice("country", "sb", places=[NIGERIA], months=last_recorded, columns=["sb_actual"])

## 6 · The three levels

A country is the sum of its admin-1 regions, which are the sum of their grid cells. The API adds up the draws first and summarises after, so a region's chance of deaths is not the sum of its cells' chances (see the data dictionary, "Adding up places or months yourself").

In [ ]:
# Nigeria's admin-1 regions in the first forecast month, from the file already downloaded.
nga_regions = first[first["country_code"] == NIGERIA]
nga_regions.nlargest(5, "sb_p_gt25")[["admin1_code", "admin1_name", "sb_p_gt25", "sb_mean"]]

In [ ]:
# Grid cells: one month, one column, the whole grid (about 65,000 cells).
cells = client.slice("grid", "sb", months=[FIRST_FORECAST_MONTH], columns=["sb_mean"])
nga_cells = cells[cells["country_code"] == NIGERIA]
print("Grid cells in Nigeria:", len(nga_cells))
nga_cells.nlargest(5, "sb_mean")[["priogrid_id", "latitude", "longitude", "admin1_name", "sb_mean"]]

## 7 · If something goes wrong

| Answer | Meaning |
|---|---|
| `401` | The key is missing, wrong, or no longer valid. |
| `404` | That level or outcome does not exist. |
| `413` | The slice asks for too much at once: ask for fewer places or months, or download the bulk file for the same level and outcome. |
| `422` | An option is wrong: a month not written `2027-01` or outside the months served, an unknown place or column, or an option given twice. |
| `503` | No forecast, or no recorded deaths, available right now. Try again later. |

## Next steps

- **`02_visualization.ipynb`**: the same data as maps.
- **`03_offline_demo.ipynb`**: the analytics on synthetic data, no key needed.
- The [partner guide](../docs/api/README.md) and the [data dictionary](../docs/api/data_dictionary.md).